## Merge `top_boundaries.py` output onto raw paragraphs

Base dataframe for everything below: paragraph content plus the rule-based TOP
boundary detector's per-paragraph output (`top_seq`, `top_number_raw`,
`top_doc_type`, `top_sponsor`).

In [ ]:
import os
import pandas as pd

data_root = os.environ['DATA_ROOT']
paras = pd.read_parquet(os.path.join(data_root, "raw/stateparl_v3_parquet/stateparl_v3_paragraphs.parquet"))
top = pd.read_parquet(os.path.join(data_root, "processed/top_boundaries.parquet"))
df = paras.merge(top, on="paragraph_id", how="left")

In [ ]:
df.describe()

,paragraph_id,period,nth,protocol_position_x,segment_position,page,protocol_position_y,top_seq
count,1.607847e+07,1.607847e+07,1.607847e+07,1.607847e+07,1.607847e+07,1.607847e+07,1.607847e+07,1.607847e+07
mean,8.085395e+06,1.317759e+01,6.181801e+01,9.983108e+02,1.701656e+01,5.151377e+01,9.983108e+02,4.438036e+00
std,4.665478e+06,5.536742e+00,3.746077e+01,7.091159e+02,2.607556e+01,3.566555e+01,7.091159e+02,4.641140e+00
min,1.000000e+00,3.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,4.052334e+06,7.000000e+00,3.100000e+01,4.310000e+02,4.000000e+00,2.400000e+01,4.310000e+02,1.000000e+00
50%,8.091652e+06,1.500000e+01,5.800000e+01,8.850000e+02,1.000000e+01,4.500000e+01,8.850000e+02,3.000000e+00
75%,1.212712e+07,1.700000e+01,8.900000e+01,1.432000e+03,2.000000e+01,7.100000e+01,1.432000e+03,7.000000e+00
max,1.616890e+07,2.200000e+01,1.700000e+02,5.938000e+03,1.053000e+03,3.940000e+02,5.938000e+03,6.300000e+01


In [ ]:
print(df.columns)
df.drop(columns=["protocol_id_y", "protocol_position_y"], inplace=True)
df.rename(columns={"protocol_id_x": "protocol_id", "protocol_position_x": "protocol_position"}, inplace=True)

In [22]:
df.head(20)

,paragraph_id,protocol_id_x,state,period,nth,date,speech_id,protocol_position_x,segment_position,page,speaker_paragraph,mandate_id,affiliation,content,protocol_id_y,protocol_position_y,top_seq,top_number_raw,top_doc_type,top_sponsor
0,1,bb_3_10,bb,3,10,2000-02-24,NaN,1,2,4,Knoblich,bb_3_pre_knoblich,pre,Werte Kolleginnen und Kollegen! Ich begrüße Si...,bb_3_10,1,0,,,
1,2,bb_3_10,bb,3,10,2000-02-24,NaN,2,3,4,,,nsc,(Allgemeiner Beifall),bb_3_10,2,0,,,
2,3,bb_3_10,bb,3,10,2000-02-24,NaN,3,4,4,Knoblich,bb_3_pre_knoblich,pre,Nicht weniger herzlich begrüße ich unsere Stam...,bb_3_10,3,0,,,
3,4,bb_3_10,bb,3,10,2000-02-24,NaN,4,5,4,Knoblich,bb_3_pre_knoblich,pre,Mit der Einladung ist Ihnen der Vorschlag zur ...,bb_3_10,4,0,,,
4,5,bb_3_10,bb,3,10,2000-02-24,NaN,5,6,4,Knoblich,bb_3_pre_knoblich,pre,"Ich darf darauf hinweisen, dass die DVU-Frakti...",bb_3_10,5,0,,,
5,6,bb_3_10,bb,3,10,2000-02-24,NaN,6,7,4,Knoblich,bb_3_pre_knoblich,pre,Damit darf ich um Ihr zustimmendes Handzeichen...,bb_3_10,6,0,,,
6,7,bb_3_10,bb,3,10,2000-02-24,NaN,7,8,4,Knoblich,bb_3_pre_knoblich,pre,"Ich merke gerade, dass es notwendig gewesen wä...",bb_3_10,7,0,,,
7,8,bb_3_10,bb,3,10,2000-02-24,NaN,8,9,4,Knoblich,bb_3_pre_knoblich,pre,"Es wird vorgeschlagen, einen neuen Tagesordnun...",bb_3_10,8,0,,,
8,9,bb_3_10,bb,3,10,2000-02-24,NaN,9,10,4,Knoblich,bb_3_pre_knoblich,pre,Außerdem wird ein neuer Tagesordnungspunkt 11 ...,bb_3_10,9,0,,,
9,10,bb_3_10,bb,3,10,2000-02-24,NaN,10,11,4,Knoblich,bb_3_pre_knoblich,pre,"Schließlich wird vorgeschlagen, einen Tagesord...",bb_3_10,10,0,,,


In [ ]:
prot = pd.read_parquet(os.path.join(data_root, "raw/stateparl_v3_parquet/stateparl_v3_protocols.parquet"))
df = df.merge(prot[["protocol_id", "url"]], on="protocol_id", how="left")

## Corpus coverage check

Protocols per state/year, to see where gaps are before drawing an annotation
sample.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, "../..")
from utils.colors import SEQ_BLUES, CHROME

prot["year"] = pd.to_datetime(prot["date"]).dt.year
counts = prot.groupby(["state", "year"]).size().unstack(fill_value=0)

all_years = list(range(int(prot["year"].min()), int(prot["year"].max()) + 1))
all_states = sorted(prot["state"].unique())
counts = counts.reindex(index=all_states, columns=all_years, fill_value=0)

ramp = [SEQ_BLUES[k] for k in sorted(SEQ_BLUES)]
cmap = mcolors.LinearSegmentedColormap.from_list("seq_blues", ramp)

fig, ax = plt.subplots(figsize=(14, 7), facecolor=CHROME["surface_light"])
ax.set_facecolor(CHROME["surface_light"])

values = counts.values.astype(float)
masked = np.ma.masked_equal(values, 0)
mesh = ax.pcolormesh(masked, cmap=cmap, edgecolors=CHROME["gridline"], linewidth=0.5)

# zero cells get a flat, distinct fill -- not the palest ramp step -- since
# telling "no protocols" apart from "one protocol" is the point of this chart
zero_rows, zero_cols = np.where(values == 0)
for r, c in zip(zero_rows, zero_cols):
    ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=CHROME["ink_muted"],
                                edgecolor=CHROME["gridline"], linewidth=0.5))

ax.set_xticks(np.arange(len(all_years)) + 0.5)
ax.set_xticklabels(all_years, rotation=90, fontsize=8, color=CHROME["ink_secondary"])
ax.set_yticks(np.arange(len(all_states)) + 0.5)
ax.set_yticklabels([s.upper() for s in all_states], fontsize=9, color=CHROME["ink_secondary"])
ax.invert_yaxis()
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title("Protocols per state by year (gray = zero)", loc="left",
             color=CHROME["ink_primary"], fontsize=13)

cbar = fig.colorbar(mesh, ax=ax, shrink=0.6, label="protocol count")
cbar.ax.yaxis.label.set_color(CHROME["ink_secondary"])
cbar.outline.set_visible(False)

plt.tight_layout()
plt.show()

## Spot-check the rule-based detector

Eyeball one random protocol per state to see if `top_boundaries.py`'s TOP
splits look sane before trusting them to build the annotation sample.

In [ ]:
def inspect_protocol(protocol_id):
    print(f"url: {df.loc[df['protocol_id'] == protocol_id, 'url'].iloc[0]}")
    tops = (
        df[(df["protocol_id"] == protocol_id) & (df["top_seq"] > 0)]
        .drop_duplicates(["protocol_id", "top_seq"])
        .sort_values("top_seq")
    )
    for _, row in tops.iterrows():
        print(f"  seq={row.top_seq} num={row.top_number_raw!r} type={row.top_doc_type!r} sponsor={row.top_sponsor!r}")

# pick a random protocol per state to spot-check
sample_ids = df.drop_duplicates("protocol_id").groupby("state")["protocol_id"].apply(lambda s: s.sample(1)).tolist()
for pid in sample_ids:
    print(f"\n===== protocol {pid} =====")
    inspect_protocol(pid)

In [ ]:
pre_cols = ["content", "affiliation", "mandate_id", "speaker_paragraph", "page", "date", "state", "protocol_id", "paragraph_id"]
pre_only = df.loc[df["affiliation"] == "pre", pre_cols]
pre_only.head(20)

## Build the annotation sample

Stratified: 3 protocols per state, spread across the early/middle/late third
of that state's date range, to catch state- and era-specific phrasing drift
rather than relying on pure random draws.

In [ ]:
# Stratified sample of protocols: 3 per state, spread across each state's date
# range (earliest/middle/latest third) rather than pure random -- catches both
# state-specific and era-specific formatting drift.
import numpy as np

rng = np.random.default_rng(42)

protocols_meta = prot.copy()
protocols_meta["date"] = pd.to_datetime(protocols_meta["date"])

sampled = []
for state, group in protocols_meta.groupby("state"):
    group = group.sort_values("date")
    n = len(group)
    for idx in np.array_split(np.arange(n), 3):
        if len(idx):
            bucket = group.iloc[idx]
            sampled.append(bucket.sample(1, random_state=rng.integers(1_000_000)))

sample_protocols = pd.concat(sampled).reset_index(drop=True)
sample_protocols[["protocol_id", "state", "date", "url"]]

In [ ]:
# All "pre" (presiding officer) paragraphs from within the sampled protocols.
# top_seq/top_number_raw/top_doc_type are kept here for the post-annotation
# comparison, but deliberately NOT shown during annotation itself (see below).
sample_ids = set(sample_protocols["protocol_id"])
annotation_cols = [
    "paragraph_id", "protocol_id", "protocol_position", "state", "date", "page",
    "mandate_id", "speaker_paragraph", "affiliation", "content",
    "top_seq", "top_number_raw", "top_doc_type", "url",
]
annotation_data = (
    df.loc[df["protocol_id"].isin(sample_ids) & (df["affiliation"] == "pre"), annotation_cols]
    .sort_values(["state", "date", "protocol_position"])
    .reset_index(drop=True)
)
annotation_data.head(20)

## Export gold-label input

Every `pre` paragraph from the sampled protocols → `top_boundaries_annotation_input.csv`,
the file `labelling/top_boundaries_annotator.py` reads.

In [ ]:
# One row per raw "pre" paragraph -- no joining of consecutive rows.
contributions = (
    annotation_data[["protocol_id", "state", "date", "url", "protocol_position", "content"]]
    .rename(columns={"protocol_position": "start_pos"})
    .copy()
)
contributions["end_pos"] = contributions["start_pos"]
contributions = contributions[["protocol_id", "state", "date", "url", "start_pos", "end_pos", "content"]]
contributions.to_csv("../../labelling/top_boundaries_annotation_input.csv", index=False)
print(f"Exported {len(contributions)} pre paragraphs from {contributions['protocol_id'].nunique()} protocols.")
print("Now run: norm_env/bin/python labelling/top_boundaries_annotator.py")